# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health survey dataset from Kilifi County, Kenya using the `mlcroissant` library. This example walks through data loading, overview, extraction, basic EDA, and visualization.

### Dataset Source
The dataset is described by a Croissant schema available at:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We use `mlcroissant` to load the dataset metadata and view the high-level description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display core metadata attributes
metadata = dataset.metadata
print("Dataset Name:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))
print("Version:", getattr(metadata, 'version', None))
print("Published:", getattr(metadata, 'datePublished', None))

## 2. Data Overview
Let us review available record sets and fields. All references to entities use their `@id` as per Croissant best practice.

In [ ]:
# List all available record sets and their IDs

record_sets = dataset.get_record_sets()
print("Record Sets available in this dataset:")
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name','N/A')}")

# For each record set, print its fields
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name','N/A')}) fields:")
    if 'field' in rs:
        for field in rs['field']:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    Field @id: {field_id}")
    else:
        print("    (No field listed)")

## 3. Data Extraction
We load data from one or more record sets into pandas DataFrames. Use record set and field `@id`s as shown above.
As an example, we extract the main survey record set and its fields.

In [ ]:
# --- You may need to adjust these record set IDs if the dataset structure is updated ---
# For this dataset, there is typically a single primary record set.
# List all record set IDs.

record_sets = dataset.get_record_sets()
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"(No records found for {record_set_id})")

# Pick the first record set as primary for demo (update this @id if you want others)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nColumns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data available for the detected record set(s).")

## 4. Exploratory Data Analysis (EDA)
Let's explore the data for initial insights. We'll filter, normalize, and group using the `@id` field names as appropriate.

Suppose the survey includes a numeric score field (e.g., `phq9_total_score` or similar, identified by its `@id`, such as 'phq9_total_score'). Adjust the field ID to one present in your data.

In [ ]:
# Set correct field IDs -- update as per your printed column list above
# For the Kilifi mental health survey, likely candidates: 'phq9_total_score', 'gad7_total_score', 'psq_total_score', etc.
# By convention, field names in the DataFrame are the '@id' attribute from the schema/CSV header.

numeric_field = None
for fieldname in [
        'phq9_total_score',
        'gad7_total_score',
        'psq_total_score'
    ]:
    if main_record_set_id and main_record_set_id in dataframes and fieldname in dataframes[main_record_set_id].columns:
        numeric_field = fieldname
        break

if numeric_field:
    threshold = 10
    df = dataframes[main_record_set_id]
    filtered_df = df[df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Example: Group by a demographic field, such as 'sex' or 'level_of_education', found via '@id'
    group_field = None
    for demo_field in ['sex', 'gender', 'level_of_education', 'marital_status']:
        if demo_field in df.columns:
            group_field = demo_field
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped_df)
    else:
        print("No suitable demographic field found for grouping.")
else:
    print("Could not find a standard numeric score field (e.g. 'phq9_total_score') in the data.")

## 5. Visualization
Now we visualize the numeric score field's distribution and grouped means. Adjust the field IDs as above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field and main_record_set_id in dataframes:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If we have a group field, show boxplot
    if group_field is not None:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mental health survey dataset for Kilifi County using only the Croissant schema URL and the `mlcroissant` library. 

**Key findings:**
- The dataset includes standardized assessments such as PHQ-9, GAD-7, and PSQ, with demographic attributes.
- We demonstrated how to filter on high symptom scores, normalize, and visualize their distribution, with grouping by demographic field.

This workflow enables reproducible, standards-based analysis using only the schema URL. For further exploration, try adjusting field names or loading additional record sets as per their `@id`.